# TP3 - ISTA, Compressed Sensing, and Matrix Completion

## Lecture Material and Results Used

- **Composite optimization**: minimize $f(x)+g(x)$ where $f$ is smooth and $g$ may be nonsmooth but has an easy proximal operator.
- **Proximal gradient / ISTA**: iterate $x_{k+1}=\operatorname{prox}_{\beta g}(x_k-\beta\nabla f(x_k))$.
- **L1 regularization**: the proximal operator of $\lambda\|x\|_1$ is soft-thresholding.
- **Compressed sensing**: sparse signals can be recovered from underdetermined linear measurements through L1-regularized objectives.
- **Matrix completion**: low-rank matrices can be recovered from partial observations through nuclear-norm regularization.
- **Singular-value thresholding**: the proximal operator of the nuclear norm shrinks singular values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(41)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})


## 1. ISTA for Sparse Recovery

We observe $y=Ax+\epsilon$ with fewer measurements than unknowns and recover a sparse vector with an L1 penalty.

In [ ]:
m, n, k = 70, 180, 10
A = rng.normal(size=(m, n)) / np.sqrt(m)
x_true = np.zeros(n)
support = rng.choice(n, size=k, replace=False)
x_true[support] = rng.normal(size=k)
y = A @ x_true + 0.025 * rng.normal(size=m)
lam = 0.04
L = np.linalg.eigvalsh(A.T @ A).max()


def soft_threshold(z, tau):
    return np.sign(z) * np.maximum(np.abs(z) - tau, 0.0)


def lasso_objective(x):
    return 0.5 * np.linalg.norm(A @ x - y) ** 2 + lam * np.abs(x).sum()


def ista(n_iter=350):
    x = np.zeros(n)
    objective, error, sparsity = [], [], []
    for _ in range(n_iter):
        grad = A.T @ (A @ x - y)
        x = soft_threshold(x - grad / L, lam / L)
        objective.append(lasso_objective(x))
        error.append(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))
        sparsity.append(np.count_nonzero(np.abs(x) > 1e-3))
    return x, np.array(objective), np.array(error), np.array(sparsity)

x_hat, obj, err, sparsity = ista()
print('relative recovery error:', err[-1].round(3))
print('estimated support size:', sparsity[-1], 'true support size:', k)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].semilogy(obj)
axes[0].set_title('ISTA objective')
axes[1].semilogy(err)
axes[1].set_title('relative recovery error')
axes[2].plot(sparsity)
axes[2].axhline(k, color='black', linestyle='--')
axes[2].set_title('active coefficients')
for ax in axes:
    ax.set_xlabel('iteration')
fig.tight_layout();

## 2. Effect of the Regularization Parameter

The L1 parameter controls the bias-sparsity tradeoff. Too small: noisy dense solution. Too large: overly sparse biased solution.

In [ ]:
grid_lam = np.logspace(-3, -0.6, 8)
summary = []
for lam_value in grid_lam:
    lam = lam_value
    x_est, _, err_curve, sparse_curve = ista(n_iter=220)
    summary.append((lam_value, err_curve[-1], sparse_curve[-1]))
lam = 0.04
summary = np.array(summary)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].semilogx(summary[:, 0], summary[:, 1], marker='o')
axes[0].set_title('recovery error')
axes[1].semilogx(summary[:, 0], summary[:, 2], marker='o')
axes[1].axhline(k, color='black', linestyle='--')
axes[1].set_title('support size')
for ax in axes:
    ax.set_xlabel('lambda')
fig.tight_layout();

## 3. Matrix Completion with Singular-Value Thresholding

The second part of TP3 concerned low-rank matrix completion. We reveal only a random subset of entries and exploit the low-rank structure.

In [ ]:
rows, cols, rank = 45, 40, 4
U = rng.normal(size=(rows, rank))
V = rng.normal(size=(cols, rank))
M_true = U @ V.T
mask = rng.random(size=M_true.shape) < 0.45
observed = np.where(mask, M_true, 0.0)


def singular_value_threshold(Y, tau):
    U, s, Vt = np.linalg.svd(Y, full_matrices=False)
    shrunk = np.maximum(s - tau, 0.0)
    return (U * shrunk) @ Vt, shrunk


def matrix_completion(n_iter=320, step=1.2, tau=0.75):
    X = np.zeros_like(M_true)
    errors, ranks, observed_errors = [], [], []
    for _ in range(n_iter):
        Y = X + step * mask * (observed - X)
        X, singular_values = singular_value_threshold(Y, tau)
        errors.append(np.linalg.norm(X - M_true) / np.linalg.norm(M_true))
        ranks.append(np.count_nonzero(singular_values > 1e-3))
        observed_errors.append(np.linalg.norm(mask * (X - M_true)) / np.linalg.norm(mask * M_true))
    return X, np.array(errors), np.array(ranks), np.array(observed_errors)

M_hat, mc_errors, mc_ranks, observed_errors = matrix_completion()
print('final matrix completion error:', mc_errors[-1].round(3))
print('final observed-entry error:', observed_errors[-1].round(3))
print('final effective rank:', mc_ranks[-1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].semilogy(mc_errors)
axes[0].set_title('relative reconstruction error')
axes[1].plot(mc_ranks, label='effective rank')
axes[1].axhline(rank, color='black', linestyle='--', label='true rank')
axes[1].set_title('effective rank under SVT')
axes[1].legend()
for ax in axes:
    ax.set_xlabel('iteration')
fig.tight_layout();

## 4. Observed Versus Recovered Matrix

The observed matrix is sparse and incomplete; the recovered matrix fills in missing entries using the low-rank prior. The reconstruction is approximate, which is exactly the useful lesson: nuclear-norm shrinkage trades data fit against low-rank structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
for ax, mat, title in zip(axes, [M_true, observed, M_hat], ['true low-rank matrix', 'observed entries', 'recovered matrix']):
    ax.imshow(mat, cmap='coolwarm', aspect='auto')
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout();

## Interpretation

In this seeded experiment the relative signal error is about 7.5%, yet the estimated support has 24 entries for 10 true nonzeros. Small spurious coefficients add little squared error while increasing the support count. For matrix completion, compare the error on observed entries with the error on the entire synthetic matrix to assess reconstruction of the missing entries.